# Delegate a task to a second agent

Have a main agent ask an auditor to check a receipt. The auditor can read the file; it cannot edit it.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/03_subagents.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a6/liteagents-0.3.0a6-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Create a receipt to check

The auditor will read this file in a temporary workspace.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="liteagents-"))
(workspace / "receipt.txt").write_text("Order A123 was paid. The total is USD 12.\n")
print((workspace / "receipt.txt").read_text())

## 4. Add an auditor

`subagents` defines and enables the auditor and its tools. The main agent gets `delegate_auditor` so it can hand over the task.

In [ ]:
from liteagents import LiteAgentClient, ProfileOptions, SubagentOptions

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.tools = ["delegate_auditor"]
profile.subagents = {
    "auditor": SubagentOptions(
        description="Check receipt.txt and report the payment status and total.",
        model=profile.model,
        tools=["read_file"],
    ),
}

## 5. Watch the delegation

You should see the auditor start and finish, then the main agent report its findings.

In [ ]:
async with LiteAgentClient(profile=profile, cwd=workspace) as agent:
    handle = await agent.start_run("Ask the auditor to verify order A123. Relay its findings.")
    async for event in handle.events():
        if event.kind in ("subagent_started", "subagent_completed"):
            print(event.kind, event.data["agent"])
    result = await handle.result()
print(result.text)

Edit the receipt and rerun the last cell. You can also give the auditor a different model through `profile.subagents["auditor"].model`.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)